# Laboratorium 7

Celem siódmego laboratorium jest zapoznanie się oraz zaimplementowanie algorytmu głębokiego uczenia aktywnego - Actor-Critic. Zaimplementowany algorytm będzie testowany z wykorzystaniem środowiska z OpenAI - *CartPole*.


Dołączenie standardowych bibliotek

In [1]:
from collections import deque
import gym
import numpy as np
import random
from torch.distributions import Categorical

if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Dołączenie bibliotek do obsługi sieci neuronowych

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

class ActorNetwork(nn.Module):
    def __init__(self, state_size, action_size, learning_rate, hidden_sizes=(128, 64)):
        super().__init__()
        h1, h2 = hidden_sizes
        self.net = nn.Sequential(
            nn.Linear(state_size, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, action_size),
            nn.Softmax(dim=-1),
        )
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        return self.net(x)

    def predict(self, state):
        self.eval()
        with torch.no_grad():
            state_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
            probs = self(state_tensor).squeeze(0)
        return probs


class CriticNetwork(nn.Module):
    def __init__(self, state_size, learning_rate, hidden_sizes=(128, 64)):
        super().__init__()
        h1, h2 = hidden_sizes
        self.net = nn.Sequential(
            nn.Linear(state_size, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, 1),
        )
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        return self.net(x)

    def predict(self, state):
        self.eval()
        with torch.no_grad():
            state_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
            value = self(state_tensor).squeeze(0)
        return value

## Zadanie 1 - Actor-Critic

<p style='text-align: justify;'>
Celem ćwiczenie jest zaimplementowanie algorytmu Actor-Critic. W tym celu należy utworzyć dwie głębokie sieci neuronowe:
    1. *actor* - sieć, która będzie uczyła się optymalnej strategii (podobna do tej z laboratorium 6),
    2. *critic* - sieć, która będzie uczyła się funkcji oceny stanu (podobnie jak się DQN).
Wagi sieci *actor* aktualizowane są zgodnie ze wzorem:
\begin{equation*}
    \theta \leftarrow \theta + \alpha \delta_t \nabla_\theta log \pi_{\theta}(a_t, s_t | \theta).
\end{equation*}
Wagi sieci *critic* aktualizowane są zgodnie ze wzorem:
\begin{equation*}
    w \leftarrow w + \beta \delta_t \nabla_w\upsilon(s_{t + 1}, w),
\end{equation*}
gdzie:
\begin{equation*}
    \delta_t \leftarrow r_t + \gamma \upsilon(s_{t + 1}, w) - \upsilon(s_t, w).
\end{equation*}
</p>

In [3]:
class ActorCriticAgent:
    def __init__(self, state_size, action_size, actor, critic, entropy_beta=0.01):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = 0.99    # discount rate
        self.learning_rate = 0.001
        self.entropy_beta = entropy_beta
        self.actor = actor
        self.critic = critic #critic network should have only one output


    def get_action(self, state):
        """
        Compute the action to take in the current state, basing on policy returned by the network.

        Note: To pick action according to the probability generated by the network
        """

        # INSERT CODE HERE to get action in a given state
        probabilities = self.actor.predict(state).cpu().numpy()
        probabilities = probabilities / probabilities.sum()
        chosen_action = np.random.choice(self.action_size, p=probabilities)
        
        return chosen_action

  

    def learn(self, state, action, reward, next_state, done):
        """
        Function learn networks using information about state, action, reward and next state. 
        First the values for state and next_state should be estimated based on output of critic network.
        Critic network should be trained based on target value:
        target = r + \gamma next_state_value if not done]
        target = r if done.
        Actor network should be trained based on delta value:
        delta = target - state_value
        """
        state_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        next_state_tensor = torch.as_tensor(next_state, dtype=torch.float32).unsqueeze(0)
        action_tensor = torch.as_tensor(action, dtype=torch.int64)
        reward_tensor = torch.as_tensor(reward, dtype=torch.float32)
        done_tensor = torch.as_tensor(done, dtype=torch.float32)

        state_value = self.critic(state_tensor).squeeze(0)
        with torch.no_grad():
            next_state_value = self.critic(next_state_tensor).squeeze(0)
            if done_tensor == 1.0:
                target_value = reward_tensor
            else:
                target_value = reward_tensor + self.gamma * next_state_value
            # target_value = reward_tensor + self.gamma * next_state_value * (1.0 - done_tensor)

        critic_loss = self.critic.loss_fn(state_value, target_value)
        self.critic.optimizer.zero_grad()
        critic_loss.backward()
        self.critic.optimizer.step()

        probs = self.actor(state_tensor).squeeze(0)
        dist = Categorical(probs=probs)
        log_prob = dist.log_prob(action_tensor)
        delta = target_value - state_value.detach()
        # entropy = dist.entropy().mean()
        # actor_loss = -(log_prob * delta) - self.entropy_beta * entropy
        actor_loss = -(log_prob * delta)

        self.actor.optimizer.zero_grad()
        actor_loss.backward()
        self.actor.optimizer.step()

<>:33: SyntaxWarning: invalid escape sequence '\g'
<>:33: SyntaxWarning: invalid escape sequence '\g'
C:\Users\jasin\AppData\Local\Temp\ipykernel_16864\2128304939.py:33: SyntaxWarning: invalid escape sequence '\g'
  target = r + \gamma next_state_value if not done]


Czas przygotować model sieci, która będzie się uczyła działania w środowisku [*CartPool*](https://gym.openai.com/envs/CartPole-v0/):

In [4]:
env = gym.make("CartPole-v0").env
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
alpha_learning_rate = 0.0001
beta_learning_rate = 0.0005

actor_model = ActorNetwork(state_size, action_size, alpha_learning_rate)
critic_model = CriticNetwork(state_size, beta_learning_rate)

d:\.Astudia\.venv\Lib\site-packages\gym\envs\registration.py:555: UserWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.warn(


Czas nauczyć agenta gry w środowisku *CartPool*:

In [5]:
agent = ActorCriticAgent(state_size, action_size, actor_model, critic_model)


for i in range(100):
    score_history = []

    for i in range(100):
        done = False
        score = 0
        # state = env.reset()
        state = env.reset()[0]
        state = torch.tensor(state, dtype=torch.float32)
        while not done:
            action = agent.get_action(state)
            next_state, reward, done, _, _ = env.step(action)
            next_state = torch.tensor(next_state, dtype=torch.float32)
            agent.learn(state, action, reward, next_state, done)
            state = next_state
            score += reward
        score_history.append(score)

    print("mean reward:%.3f" % (np.mean(score_history)))

    if np.mean(score_history) > 300:
        print("You Win!")
        break

d:\.Astudia\.venv\Lib\site-packages\torch\nn\modules\loss.py:634: UserWarning: Using a target size (torch.Size([])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


mean reward:19.050
mean reward:40.270
mean reward:129.660
mean reward:479.510
You Win!
